# Multi-Agent Router Architecture: Design, Routing, and Human-in-the-Loop

---

## Learning Objectives

By the end of this notebook, you will be able to:

1. Explain why **multi-agent router architectures** outperform monolithic agents on broad task spaces.
2. Build a **structured routing layer** using Pydantic and `with_structured_output()` to classify user intent.
3. Implement **Human-in-the-Loop (HITL) gates** that let users confirm or override automated decisions.
4. Understand how **specialist supervisors** (Knowledge, Code, Engineering, Chat) handle domain-specific tasks.
5. Design **memory and context management** strategies including checkpointing and conversation summarization.

---

## Why Multi-Agent?

A single general-purpose agent struggles when the task space is too broad. It must simultaneously be good at
factual retrieval, code generation, technical explanation, and casual conversation. Each domain has different
tool requirements, prompting strategies, and evaluation criteria.

The **router architecture** solves this by decomposing the problem: a lightweight classifier determines user
intent and delegates to a specialist agent that is purpose-built for that domain. Each specialist can have
its own tools, state, sub-agents, and approval flows.

---

## Architecture Overview

```
User Input
    |
    v
+---------------+
|    Router      | <-- Classifies intent using structured output
|   (Pydantic)   |
+-------+-------+
        |
        v
+----------------+   HITL Gate 1: Confirm or override routing
| Routing Gate   |
+-------+--------+
        |
        +---------------+---------------+---------------+
        |               |               |               |
        v               v               v               v
   Knowledge         Code          Engineering        Chat
   Supervisor      Supervisor      Supervisor      Supervisor
   (RAG+Web)      (Gen+Exec)     (Sub-agents)   (Conversation)
        |               |
   FAISS+Tavily    HITL Gate 2:
   in parallel     Approve code
```

This notebook walks through each layer of this architecture. The **full runnable system** lives in
`agent-patterns/Langgraph/` and `agent-patterns/MAF/` -- here we focus on understanding the design
decisions and demonstrating the key components in isolation.

---

### How This Differs from Notebook 06 (Orchestrator-Worker)

| Aspect | Orchestrator-Worker (Notebook 06) | Multi-Agent Router (This Notebook) |
|--------|-----------------------------------|------------------------------------|
| **Routing** | Orchestrator creates subtasks dynamically | Router classifies intent into fixed supervisor categories |
| **Workers** | Stateless, single-turn workers | Stateful supervisors with their own sub-graphs |
| **HITL** | None | Two gates: routing approval + code approval |
| **Persistence** | None | SQLite checkpointing, conversation summarization |
| **Session** | Single request-response | Multi-turn conversation with memory |

In [1]:
# ---- Setup ----

import os
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_openai import AzureChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

load_dotenv()

llm = AzureChatOpenAI(
    api_version="2024-12-01-preview",
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    model=os.getenv("AZURE_OPENAI_MODEL", "gpt-4.1-mini"),
    temperature=0,
)

print(f"LLM initialized: {llm.model}")

LLM initialized: gpt-4.1-mini


## Section 2: Structured Routing with Pydantic

The router is the entry point of the system. It receives the user's message and must decide which
specialist supervisor should handle it. Rather than parsing free-text output, we use
`with_structured_output()` to force the LLM to return a Pydantic model.

This gives us:
- **Type safety** -- the supervisor field is always a string, confidence is always a float
- **Guaranteed structure** -- no regex parsing, no JSON extraction hacks
- **Reasoning transparency** -- the model must explain its routing decision

In the production system (`Router.py`), the `RoutingDecision` model has two fields:
- `supervisor`: one of `knowledge`, `code`, `chat`, `engineering`
- `reason`: a one-sentence explanation

Here we add a `confidence` field to demonstrate how you might build confidence-based HITL gates.

In [2]:
# ---- Pydantic model for structured routing decisions ----

class RoutingDecision(BaseModel):
    """Structured output from the routing LLM."""
    supervisor: str = Field(
        description="Exactly one of: 'knowledge', 'code', 'chat', 'engineering'"
    )
    confidence: float = Field(
        description="Confidence score between 0.0 and 1.0"
    )
    reasoning: str = Field(
        description="One sentence explaining why this routing was chosen"
    )


# Bind the Pydantic model to the LLM -- every call now returns a RoutingDecision
router_chain = llm.with_structured_output(RoutingDecision)

print("Router chain created with structured output.")
print(f"Output schema: {RoutingDecision.model_json_schema()['properties'].keys()}")

Router chain created with structured output.
Output schema: dict_keys(['supervisor', 'confidence', 'reasoning'])


In [3]:
# ---- Routing prompt (mirrors the production Router.py) ----

ROUTING_SYSTEM_PROMPT = """You are a router directing user queries to the right AI supervisor.

Supervisors:
- knowledge    -> factual questions, research, document lookup, web search
- code         -> write/run/fix Python code
- engineering  -> explanations, architecture, tradeoffs, best practices, system design,
                  memory flow, framework comparisons, or technical reasoning
                  (output is mainly explanation, not code)
- chat         -> conversation, brainstorming, follow-ups, anything else

Rules:
- If the user wants factual information or research, prefer knowledge.
- If the user wants code written, fixed, debugged, or executed, prefer code.
- If the user is asking for explanation of architecture, system design, or technical
  reasoning, prefer engineering.
- Otherwise prefer chat.
- Set confidence to reflect how clearly the query maps to one supervisor.
"""


def route_query(user_message: str) -> RoutingDecision:
    """Route a single user message to the appropriate supervisor."""
    prompt = f"{ROUTING_SYSTEM_PROMPT}\nUser message: {user_message}"
    return router_chain.invoke(prompt)


# ---- Test with several queries ----
test_queries = [
    "What is the capital of France?",
    "Write a Python function to sort a list using merge sort",
    "Explain the difference between microservices and monoliths",
    "How are you doing today?",
]

print(f"{'Query':<58} {'Supervisor':<14} {'Confidence':<12} Reasoning")
print("-" * 120)

for query in test_queries:
    decision = route_query(query)
    print(f"{query:<58} {decision.supervisor:<14} {decision.confidence:<12.2f} {decision.reasoning}")

# Notice how each query maps cleanly to a different supervisor. The structured
# output guarantees we can branch on the result programmatically.
# In production, the routing prompt also includes conversation history so the
# router can handle follow-up queries.


Query                                                      Supervisor     Confidence   Reasoning
------------------------------------------------------------------------------------------------------------------------


/Users/josephazar/Desktop/WORK/GITHUB/agentic-ai-langchain-maf-workshop/venv/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=RoutingDecision(superviso... knowledge supervisor.'), input_type=RoutingDecision])
  return self.__pydantic_serializer__.to_python(


What is the capital of France?                             knowledge      0.95         The user is asking a factual question about the capital of France, which fits the knowledge supervisor.


/Users/josephazar/Desktop/WORK/GITHUB/agentic-ai-langchain-maf-workshop/venv/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=RoutingDecision(superviso...hich is a coding task.'), input_type=RoutingDecision])
  return self.__pydantic_serializer__.to_python(


Write a Python function to sort a list using merge sort    code           0.95         The user explicitly requests writing a Python function to implement merge sort, which is a coding task.


/Users/josephazar/Desktop/WORK/GITHUB/agentic-ai-langchain-maf-workshop/venv/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=RoutingDecision(superviso...ngineering supervisor.'), input_type=RoutingDecision])
  return self.__pydantic_serializer__.to_python(


Explain the difference between microservices and monoliths engineering    0.95         The user is asking for an explanation of architectural styles, which falls under system design and technical reasoning best handled by the engineering supervisor.


How are you doing today?                                   chat           0.90         The user is engaging in casual conversation, which fits best with the chat supervisor.


/Users/josephazar/Desktop/WORK/GITHUB/agentic-ai-langchain-maf-workshop/venv/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=RoutingDecision(superviso...h the chat supervisor.'), input_type=RoutingDecision])
  return self.__pydantic_serializer__.to_python(


## Section 3: Human-in-the-Loop (HITL) Gates

Automated routing is powerful but not infallible. A user asking "write me a comparison of React vs Vue"
could reasonably go to `engineering` (technical reasoning) or `code` (generate example code). HITL gates
let the user confirm or override the router's decision before any work begins.

This architecture has **two HITL gates**:

### Gate 1: Routing Approval
After the router classifies intent, the user sees the proposed supervisor and reason. They can:
- Press Enter to confirm
- Type a different supervisor name to override

### Gate 2: Code Approval
When the code supervisor generates code, it is presented to the user **before execution**. They can:
- Type `approve` to run the code as-is
- Type `cancel` to abort
- Paste edited code to run their modified version

---

### LangGraph vs MAF: Different HITL Mechanisms

**LangGraph** uses `interrupt()` and `Command(resume=)`. When a node calls `interrupt()`, the graph
execution pauses and returns control to the caller. The graph state is persisted to the SQLite
checkpointer. When the caller invokes the graph again with `Command(resume=value)`, execution
resumes from exactly where it paused.

**MAF** uses Python's built-in `input()`. Since MAF agents are async functions, the code simply
blocks on `input()` in the main loop. This is simpler but cannot persist state across process
restarts.

```
LangGraph HITL Flow:

  graph.invoke(input)           # starts execution
       |                        
       v                        
  [router_node]                 
       |                        
       v                        
  [routing_gate_node]           
       |                        
   interrupt({                  # graph PAUSES here
     "proposed_supervisor": ...,
     "reason": ...             
   })                           
       |                        
       v   <-- state saved to SQLite
  (control returns to caller)   
                                
  graph.invoke(                 # caller resumes
    Command(resume="ok")        
  )                             
       |                        
       v                        
  [dispatch_node]  <-- continues from where it paused
```

In [4]:
# ---- Demonstrating the HITL routing gate pattern ----
#
# In the production system, this is a LangGraph node that calls interrupt().
# Here we show the logic as a standalone function to illustrate the concept.

VALID_SUPERVISORS = {"knowledge", "code", "chat", "engineering"}


def routing_gate(proposed: str, reason: str, human_input: str = "") -> str:
    """
    HITL Gate 1: Routing approval.
    
    In production, `human_input` comes from interrupt()/Command(resume=).
    Here we pass it as a parameter to make it testable in a notebook.
    
    Returns the confirmed supervisor name.
    """
    human_input = human_input.strip().lower()
    
    # Empty or affirmative = accept the router's decision
    if not human_input or human_input in ("ok", "yes", "y", proposed):
        return proposed
    
    # Valid override
    if human_input in VALID_SUPERVISORS:
        return human_input
    
    # Unrecognized input = keep original
    return proposed


# Test the gate with different human responses
test_cases = [
    ("knowledge", "",            "User presses Enter"),
    ("knowledge", "ok",          "User types 'ok'"),
    ("knowledge", "engineering", "User overrides to engineering"),
    ("code",      "chat",        "User overrides code -> chat"),
    ("code",      "gibberish",   "Unrecognized input keeps original"),
]

print(f"{'Proposed':<14} {'Human Input':<16} {'Result':<14} Scenario")
print("-" * 75)
for proposed, human_input, scenario in test_cases:
    result = routing_gate(proposed, "test reason", human_input)
    print(f"{proposed:<14} {repr(human_input):<16} {result:<14} {scenario}")

Proposed       Human Input      Result         Scenario
---------------------------------------------------------------------------
knowledge      ''               knowledge      User presses Enter
knowledge      'ok'             knowledge      User types 'ok'
knowledge      'engineering'    engineering    User overrides to engineering
code           'chat'           chat           User overrides code -> chat
code           'gibberish'      code           Unrecognized input keeps original


## Section 4: The Knowledge Supervisor

The knowledge supervisor is itself a **mini-orchestrator**. When a factual query arrives, it must
decide whether to search internal documents (RAG via FAISS), the web (Tavily), or both.

### Internal Architecture

```
User Question
      |
      v
+------------------+
|   Supervisor     |  Decides which agents to call
|  (KnowledgePlan) |  using structured output
+--------+---------+
         |
    +----+----+
    |         |
    v         v
 [RAG]     [Web]       Both run in PARALLEL via asyncio.gather
  FAISS    Tavily
    |         |
    +----+----+
         |
         v
   +----------+
   |  Merger   |  Combines results into a coherent answer
   +----------+
```

The key design decisions:

1. **Structured planning**: The supervisor uses a `KnowledgePlan` Pydantic model with two `AgentTask`
   fields (rag and web). Each task can be `null` to skip that source.

2. **Parallel execution**: RAG and web search are independent operations. Running them concurrently
   with `asyncio.gather` cuts latency nearly in half compared to sequential execution.

3. **Result merging**: A dedicated merger node receives both results and produces a unified answer,
   clearly attributing information to its source.

In [5]:
# ---- Parallel execution pattern from knowledge_agent.py ----
#
# This demonstrates the async pattern used to run RAG and web search concurrently.
# In the production system, rag_agent() queries FAISS and web_agent() calls Tavily.

import asyncio
import time


async def mock_rag_search(query: str) -> str:
    """Simulates a FAISS vector store retrieval."""
    await asyncio.sleep(1.0)  # simulate latency
    return f"[RAG] Found 4 relevant documents for: {query}"


async def mock_web_search(query: str) -> str:
    """Simulates a Tavily web search."""
    await asyncio.sleep(1.2)  # simulate latency
    return f"[Web] Found 5 search results for: {query}"


async def run_knowledge_query(question: str) -> dict:
    """Run RAG and web search in parallel, exactly as knowledge_agent.py does."""
    start = time.time()
    
    # Both coroutines start at the same time
    rag_result, web_result = await asyncio.gather(
        mock_rag_search(question),
        mock_web_search(question),
    )
    
    elapsed = time.time() - start
    return {
        "rag": rag_result,
        "web": web_result,
        "elapsed": elapsed,
    }


# Run the parallel query
result = await run_knowledge_query("What is quantum computing?")

print(f"RAG result : {result['rag']}")
print(f"Web result : {result['web']}")
print(f"Total time : {result['elapsed']:.2f}s (both ran concurrently, not 2.2s sequential)")

RAG result : [RAG] Found 4 relevant documents for: What is quantum computing?
Web result : [Web] Found 5 search results for: What is quantum computing?
Total time : 1.20s (both ran concurrently, not 2.2s sequential)


## Section 5: The Code Supervisor

The code supervisor has the most complex pipeline of any specialist, because code execution is
inherently risky and requires human oversight.

### Pipeline Architecture

```
User Request
      |
      v
+------------------+
|   Supervisor     |  Generates code using structured output (CodeBlock)
+--------+---------+
         |
         v
+------------------+
|  HITL: Approve?  |  User reviews generated code
+--------+---------+
    /    |    \
   v     v     v
 Approve Cancel Edit
   |             |
   v             v
+----------+  [Revise and re-approve]
| Execute  |
+----+-----+
     |
     +-----> Success --> [Finalizer] --> Return result
     |
     +-----> Error ----> [Debugger] --> [Re-execute] (up to 3 retries)
```

### Key Design Decisions

1. **Structured code generation**: The LLM returns a `CodeBlock` Pydantic model with separate `code`
   and `explanation` fields. This prevents markdown/backtick contamination in the executable code.

2. **Pre-execution checks**: Before running code, the system:
   - Extracts imports via AST parsing
   - Auto-installs missing pip packages
   - Checks for missing environment variables

3. **Sandboxed execution**: Code runs in a subprocess with a timeout (30s default), isolating it
   from the main process.

4. **Auto-debugging**: If execution fails, a debug LLM analyzes the error and all previous attempts,
   then produces a `DebugResult` with corrected code. This loops up to `MAX_RETRIES` (3) times.

In [6]:
# ---- Structured code generation schema (from code_agent.py) ----

class CodeBlock(BaseModel):
    """Structured output for generated code."""
    code: str = Field(description="Complete runnable Python code. No markdown, no backticks.")
    explanation: str = Field(description="Brief explanation of what the code does.")


class DebugResult(BaseModel):
    """Structured output for debugging failed code."""
    fixed_code: str = Field(description="Corrected Python code. No markdown, no backticks.")
    explanation: str = Field(description="What was wrong and what was fixed.")
    gave_up: bool = Field(default=False, description="True only if the error is fundamentally unsolvable.")


# Generate code using structured output
code_llm = llm.with_structured_output(CodeBlock)

result = code_llm.invoke(
    "Write a Python function that checks if a string is a palindrome. "
    "Include test cases with print statements."
)

print("--- Generated Code ---")
print(result.code)
print("\n--- Explanation ---")
print(result.explanation)
print("\n--- Code is clean (no backticks, no markdown) ---")
print(f"Starts with backtick: {'```' in result.code}")

--- Generated Code ---
def is_palindrome(s):
    # Remove spaces and convert to lowercase for uniformity
    s = s.replace(' ', '').lower()
    # Check if the string is equal to its reverse
    return s == s[::-1]

# Test cases
print(is_palindrome('racecar'))  # True
print(is_palindrome('hello'))    # False
print(is_palindrome('A man a plan a canal Panama'))  # True
print(is_palindrome('No lemon no melon'))  # True
print(is_palindrome('Python'))   # False

--- Explanation ---
This function checks if a given string is a palindrome by removing spaces, converting it to lowercase, and comparing it to its reverse. The test cases demonstrate the function with various inputs, including phrases with spaces and mixed cases.

--- Code is clean (no backticks, no markdown) ---
Starts with backtick: False


/Users/josephazar/Desktop/WORK/GITHUB/agentic-ai-langchain-maf-workshop/venv/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=CodeBlock(code="def is_pa...paces and mixed cases.'), input_type=CodeBlock])
  return self.__pydantic_serializer__.to_python(


In [7]:
# ---- Simulating the code approval flow ----
#
# In production, this is a LangGraph node with interrupt().
# Here we show the branching logic.

def code_approval_gate(code: str, human_input: str) -> dict:
    """
    HITL Gate 2: Code approval.
    
    Returns a dict describing what should happen next.
    In production, this maps to LangGraph state updates.
    """
    human_input = human_input.strip()
    
    if human_input.lower() == "cancel":
        return {"action": "cancelled", "code": None}
    
    if human_input.lower() == "approve":
        return {"action": "execute", "code": code}
    
    # User pasted modified code
    return {"action": "execute", "code": human_input}


# Demonstrate the three paths
sample_code = "print('Hello, World!')"

for human_input, label in [
    ("approve",                      "Approve as-is"),
    ("cancel",                       "Cancel execution"),
    ("print('Modified by user!')",   "User provides edited code"),
]:
    result = code_approval_gate(sample_code, human_input)
    print(f"  {label:<30} -> action={result['action']}, code={repr(result['code'])[:50]}")

  Approve as-is                  -> action=execute, code="print('Hello, World!')"
  Cancel execution               -> action=cancelled, code=None
  User provides edited code      -> action=execute, code="print('Modified by user!')"


## Section 6: The Engineering Supervisor

The engineering supervisor handles technical explanations, architecture questions, and framework
comparisons. It is itself a multi-agent system with three sub-agents:

- **Research agent** -- produces explanations and conceptual content
- **Code agent** -- generates illustrative code snippets
- **Comparison agent** -- creates structured side-by-side comparisons

A supervisor node uses a `SupervisorPlan` Pydantic model to decide which sub-agents are needed.
For example, "Explain microservices vs monoliths" would activate the research and comparison agents
but skip the code agent.

Each sub-agent maintains a **memory list** persisted via the SQLite checkpointer, so follow-up
questions can reference previous outputs without repeating them.

```
+------------------+
|   Supervisor     |  Decides which sub-agents to call (SupervisorPlan)
+--------+---------+
         |
    +----+----+----+
    |    |    |    |
    v    v    v    |
 [Res] [Code] [Cmp]|  Fan-out: only activated agents run
    |    |    |    |
    +----+----+----+
         |
         v
   +----------+
   |  Merger   |  Combines outputs from all activated agents
   +----------+
```

## Section 7: Memory and Context Management

A multi-turn router system accumulates conversation history across many specialist agents.
Without management, the context window fills up and performance degrades. This architecture
uses two strategies:

### Strategy 1: State Persistence

| Mechanism | LangGraph | MAF |
|-----------|-----------|-----|
| **Storage** | `SqliteSaver` checkpointer | JSON files via `memory_store.py` |
| **Scope** | Automatic -- every node's state is saved | Manual -- explicit `load_session`/`save_session` |
| **Resume** | Can resume mid-graph after process restart | Must replay from saved history |
| **HITL** | Graph pauses and resumes seamlessly | Blocks on `input()`, no cross-restart resume |

The LangGraph approach (`SqliteSaver`) is more robust: if the process crashes after the routing
gate but before dispatch, the graph can resume from the exact interruption point on restart.
The MAF approach is simpler to implement and debug.

### Strategy 2: Conversation Summarization

When conversation history exceeds a threshold (10 messages in the production system), the router
summarizes older messages and keeps only the most recent 3 turns in full. This prevents
context window overflow while preserving important context.

In [8]:
# ---- Conversation summarization pattern (from Router.py) ----

def summarize_if_needed(
    messages: list,
    threshold: int = 10,
    recent_keep: int = 3,
) -> list:
    """
    If conversation exceeds threshold, summarize older messages and keep
    only the most recent `recent_keep` messages in full.
    
    This mirrors the logic in Router.py's router_node() and
    MAF Router.py's maybe_summarize().
    """
    if len(messages) <= threshold:
        return messages  # no summarization needed
    
    # Split into old (to summarize) and recent (to keep)
    old_messages = messages[:-recent_keep]
    recent_messages = messages[-recent_keep:]
    
    # Build summary text from old messages
    summary_parts = []
    for msg in old_messages:
        if isinstance(msg, HumanMessage):
            summary_parts.append(f"User: {msg.content}")
        elif isinstance(msg, AIMessage):
            summary_parts.append(f"Assistant: {msg.content}")
    
    summary_text = "\n".join(summary_parts)
    
    # In production, this would call the LLM to produce a concise summary.
    # Here we show the structure.
    summary_prompt = f"Summarize this conversation concisely:\n{summary_text}"
    summary_response = llm.invoke([HumanMessage(content=summary_prompt)])
    
    # Replace old messages with summary + keep recent messages
    summarized = [
        AIMessage(content=f"Previous conversation summary: {summary_response.content}")
    ] + recent_messages
    
    return summarized


# Demonstrate with a long conversation
long_conversation = []
for i in range(12):
    long_conversation.append(HumanMessage(content=f"User message {i+1}"))
    long_conversation.append(AIMessage(content=f"Assistant response {i+1}"))

print(f"Original conversation length : {len(long_conversation)} messages")

trimmed = summarize_if_needed(long_conversation, threshold=10, recent_keep=3)
print(f"After summarization          : {len(trimmed)} messages")
print(f"First message type           : summary")
print(f"Remaining messages           : last 3 from original conversation")

Original conversation length : 24 messages


After summarization          : 4 messages
First message type           : summary
Remaining messages           : last 3 from original conversation


## Section 8: Full System Walkthrough

Let us trace an end-to-end multi-turn session to see how all components work together.
This simulates three consecutive user messages that each route to a different supervisor.

### Turn 1: Factual Question
```
User: "What is quantum computing?"
  -> Router: knowledge (confidence: 0.95)
  -> HITL Gate 1: User confirms
  -> Knowledge Supervisor:
       -> RAG: searches internal docs (4 chunks)
       -> Web: searches Tavily (5 results)       } parallel
       -> Merger: combines into coherent answer
  -> Answer returned to user
```

### Turn 2: Code Request (references Turn 1)
```
User: "Write me a Python simulation of a qubit"
  -> Router: code (confidence: 0.92)
     (Router sees Turn 1 context, understands the domain)
  -> HITL Gate 1: User confirms
  -> Code Supervisor:
       -> Generates CodeBlock via structured output
       -> HITL Gate 2: User reviews code, types "approve"
       -> Executes in subprocess sandbox
       -> Success on attempt 1
  -> Code output returned to user
```

### Turn 3: Casual Follow-up
```
User: "Thanks, that was really helpful"
  -> Router: chat (confidence: 0.98)
  -> HITL Gate 1: User confirms
  -> Chat Supervisor:
       -> Receives conversation summary from router
       -> Generates friendly response referencing the quantum topic
  -> Response returned to user
```

Notice how **context flows across supervisors**. The router maintains conversation history and
passes a trimmed version to each specialist. The code supervisor knows about quantum computing
from Turn 1 because the router's conversation summary includes it.

In [9]:
# ---- Simulating a multi-turn session ----
#
# This demonstrates how the router classifies different intents across turns,
# showing that context from previous turns influences routing decisions.

ROUTING_PROMPT_WITH_HISTORY = """You are a router directing user queries to the right AI supervisor.

Supervisors:
- knowledge    -> factual questions, research, document lookup, web search
- code         -> write/run/fix Python code
- engineering  -> explanations, architecture, tradeoffs, system design
- chat         -> conversation, brainstorming, follow-ups, anything else

Recent conversation:
{conversation_history}

Latest user message:
{user_message}

Rules:
- If the user refers to "that", "it", "this", use the conversation to understand what they mean.
- Set confidence to reflect how clearly the query maps to one supervisor.
"""

conversation_history = []
session_turns = [
    "What is quantum computing?",
    "Write me a Python simulation of a qubit",
    "Thanks, that was really helpful",
]

for user_msg in session_turns:
    history_text = "\n".join(conversation_history) if conversation_history else "No previous conversation."
    
    prompt = ROUTING_PROMPT_WITH_HISTORY.format(
        conversation_history=history_text,
        user_message=user_msg,
    )
    
    decision = router_chain.invoke(prompt)
    
    print(f"User: {user_msg}")
    print(f"  -> {decision.supervisor.upper()} (confidence: {decision.confidence:.2f})")
    print(f"     {decision.reasoning}")
    print()
    
    # Update history (simulating what the router does)
    conversation_history.append(f"User: {user_msg}")
    conversation_history.append(f"Assistant: [Response from {decision.supervisor} supervisor]")

User: What is quantum computing?
  -> KNOWLEDGE (confidence: 0.95)
     The user is asking a factual question about quantum computing, which fits best with the knowledge supervisor.



/Users/josephazar/Desktop/WORK/GITHUB/agentic-ai-langchain-maf-workshop/venv/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=RoutingDecision(superviso...y the code supervisor.'), input_type=RoutingDecision])
  return self.__pydantic_serializer__.to_python(


User: Write me a Python simulation of a qubit
  -> CODE (confidence: 0.95)
     The user requests a Python simulation of a qubit, which is a coding task best handled by the code supervisor.



User: Thanks, that was really helpful
  -> CHAT (confidence: 0.90)
     The user is expressing gratitude and continuing the conversation without a specific factual or coding request, so routing to chat is appropriate.



/Users/josephazar/Desktop/WORK/GITHUB/agentic-ai-langchain-maf-workshop/venv/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=RoutingDecision(superviso...o chat is appropriate.'), input_type=RoutingDecision])
  return self.__pydantic_serializer__.to_python(


## Section 9: LangGraph State and Graph Construction

The production router uses LangGraph's `StateGraph` to define the flow between nodes.
The state is a `TypedDict` with fields shared across all nodes:

```python
class RouterState(TypedDict):
    messages: Annotated[list, add_messages]   # conversation history
    raw_messages: Annotated[list, add_messages]
    active_supervisor: str                     # which supervisor to dispatch to
    routing_reason: str                        # why the router chose this
    final_answer: str                          # result from the specialist
    session_id: str                            # for checkpointer keying
    pending_code_approval: bool                # signals HITL Gate 2
    pending_code_message: str                  # user's approval decision
    conversation_summary: str                  # rolling summary
```

The graph topology uses **conditional edges** to implement branching:

```python
workflow = StateGraph(RouterState)

workflow.add_node("router",        router_node)
workflow.add_node("routing_gate",  routing_gate_node)     # HITL 1
workflow.add_node("dispatch",      dispatch_node)
workflow.add_node("approval_gate", approval_gate_node)    # HITL 2

workflow.add_edge(START, "router")
workflow.add_edge("router", "routing_gate")
workflow.add_edge("routing_gate", "dispatch")

# After dispatch: if code needs approval, go to approval_gate; otherwise END
workflow.add_conditional_edges(
    "dispatch",
    route_after_dispatch,
    {"approval_gate": "approval_gate", END: END}
)

# After approval: always loop back to dispatch to execute the code
workflow.add_conditional_edges(
    "approval_gate",
    route_after_approval_gate,
    {"dispatch": "dispatch"}
)

graph = workflow.compile(checkpointer=SqliteSaver(conn))
```

The `SqliteSaver` checkpointer means this graph survives process restarts. A session ID
(UUID) is used as the `thread_id` to isolate different users' conversations.

## Section 10: Summary and Comparison

### LangGraph vs MAF for Multi-Agent Systems

| Aspect | LangGraph | MAF |
|--------|-----------|-----|
| **State management** | `TypedDict` + `SqliteSaver` checkpointer | JSON files via `memory_store.py` |
| **HITL mechanism** | `interrupt()` + `Command(resume=)` | `input()` in async main loop |
| **Routing** | Conditional edges on `StateGraph` | `async` dispatch function with if/elif |
| **Parallelism** | Optional (nodes can use `asyncio`) | Native async (`asyncio.gather`) |
| **Resumability** | Full -- survives process restart | Partial -- history persists, mid-flow state lost |
| **Complexity** | Higher -- graph DSL, state reducers | Lower -- plain Python async functions |
| **Debugging** | Graph visualization, state snapshots | Log files, print statements |

Both implementations achieve the same user-facing behavior: structured routing, HITL gates,
specialist agents, and conversation memory. The choice depends on your requirements for
persistence, resumability, and team familiarity with the frameworks.

---

### References

The full production implementations are available in this repository:

- **LangGraph**: `agent-patterns/Langgraph/Router.py`, `knowledge_agent.py`, `code_agent.py`,
  `engineering_agent.py`, `chat_agent.py`
- **MAF**: `agent-patterns/MAF/Router.py`, `knowledge_agent.py`, `code_agent.py`,
  `engineering_agent.py`, `chat_agent.py`

---

### Exercises

1. **Add a 5th supervisor**: Create a "data analysis" supervisor that handles pandas/SQL queries.
   Update the `RoutingDecision` model, the routing prompt, and the dispatch logic.

2. **Confidence-based HITL**: Modify the routing gate so it only asks for confirmation when
   `confidence < 0.8`. High-confidence routing should proceed automatically.

3. **Implement conversation summarization**: Take the `summarize_if_needed()` function from
   Section 7 and integrate it into a multi-turn loop that maintains a rolling summary.